In [1]:
import csv
 
 
E_MG_BULK = -1.54928
E_ZN_BULK = -1.09020
 
 
STABLE = {
    "MgZn2":    {"n_Mg": 4,   "n_Zn": 8,   "E_total": -16.195525},
    "Mg4Zn7":   {"n_Mg": 40,  "n_Zn": 70,  "E_total": -149.461242},
    "Mg2Zn11":  {"n_Mg": 6,   "n_Zn": 33,  "E_total":  -45.759800},
    "Mg21Zn25": {"n_Mg": 126, "n_Zn": 150, "E_total": -383.303370},
    "Mg51Zn20": {"n_Mg": 102, "n_Zn": 40,  "E_total": -205.610584},
}
 
 
DFT_REF = {
    "MgZn2":    -0.0990,
    "Mg4Zn7":   -0.0870,
    "Mg2Zn11":  -0.0420,
    "Mg21Zn25": -0.0810,
    "Mg51Zn20": None,
}
 
EXP_REF = {
    "MgZn2":    -0.0550,
    "Mg4Zn7":   None,
    "Mg2Zn11":  None,
    "Mg21Zn25": None,
    "Mg51Zn20": None,
}
 
JANG_REF = {
    "MgZn2":    -0.1020,
    "Mg4Zn7":   None,
    "Mg2Zn11":  None,
    "Mg21Zn25": -0.0960,
    "Mg51Zn20": None,
}
 
 
def formation_enthalpy(n_Mg, n_Zn, E_total):
    n = n_Mg + n_Zn
    return (E_total - n_Mg * E_MG_BULK - n_Zn * E_ZN_BULK) / n
 
 
def write_csv(filename, fieldnames, rows):
    with open(filename, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)
    print(f"Wrote {filename}")
 
 
def main():
    rows = []
    for name, d in STABLE.items():
        n_total = d["n_Mg"] + d["n_Zn"]
        dH = formation_enthalpy(d["n_Mg"], d["n_Zn"], d["E_total"])
        rows.append({
            "phase":                name,
            "x_Zn":                 round(d["n_Zn"] / n_total, 3),
            "atoms_per_cell":       n_total,
            "E_total_eV":           round(d["E_total"], 4),
            "deltaH_f_eV_per_atom": round(dH, 4),
        })
    rows.sort(key=lambda r: r["deltaH_f_eV_per_atom"])
 
    write_csv("results_stable.csv",
              ["phase", "x_Zn", "atoms_per_cell", "E_total_eV", "deltaH_f_eV_per_atom"],
              rows)
 
    comparison = []
    for r in rows:
        phase = r["phase"]
        meam = r["deltaH_f_eV_per_atom"]
        dft = DFT_REF.get(phase)
        dev = (meam - dft) / dft * 100 if dft is not None else None
        comparison.append({
            "phase":          phase,
            "MEAM":           meam,
            "DFT_Du2023":     dft if dft is not None else "-",
            "Experimental":   EXP_REF.get(phase) if EXP_REF.get(phase) is not None else "-",
            "Jang_MEAM_2018": JANG_REF.get(phase) if JANG_REF.get(phase) is not None else "-",
            "dev_vs_DFT_pct": f"{dev:+.1f}" if dev is not None else "-",
        })
 
    write_csv("comparison_vs_DFT.csv",
              ["phase", "MEAM", "DFT_Du2023", "Experimental",
               "Jang_MEAM_2018", "dev_vs_DFT_pct"],
              comparison)
 
    print("\nStable intermetallic phases (eV/atom):")
    for r in rows:
        print(f"  {r['phase']:10s}  x_Zn={r['x_Zn']:.3f}  "
              f"DeltaH_f = {r['deltaH_f_eV_per_atom']:+.4f}")
 
 
if __name__ == "__main__":
    main()

Wrote results_stable.csv
Wrote comparison_vs_DFT.csv

Stable intermetallic phases (eV/atom):
  MgZn2       x_Zn=0.667  DeltaH_f = -0.1064
  Mg4Zn7      x_Zn=0.636  DeltaH_f = -0.1016
  Mg21Zn25    x_Zn=0.543  DeltaH_f = -0.0890
  Mg51Zn20    x_Zn=0.282  DeltaH_f = -0.0280
  Mg2Zn11     x_Zn=0.846  DeltaH_f = -0.0125


In [2]:
import csv
 
 
E_MG_BULK = -1.54928
E_ZN_BULK = -1.09020
 
 
ANTISITE = {
    "ZnMg2":   {"n_Mg": 8,  "n_Zn": 4,  "E_total":  -11.907042, "parent": "MgZn2"},
    "Zn2Mg11": {"n_Mg": 33, "n_Zn": 6,  "E_total":  -33.440644, "parent": "Mg2Zn11"},
    "Zn4Mg7":  {"n_Mg": 70, "n_Zn": 40, "E_total": -150.913624, "parent": "Mg4Zn7"},
}
 
 
def formation_enthalpy(n_Mg, n_Zn, E_total):
    n = n_Mg + n_Zn
    return (E_total - n_Mg * E_MG_BULK - n_Zn * E_ZN_BULK) / n
 
 
def write_csv(filename, fieldnames, rows):
    with open(filename, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)
    print(f"Wrote {filename}")
 
 
def main():
    rows = []
    for name, d in ANTISITE.items():
        dH = formation_enthalpy(d["n_Mg"], d["n_Zn"], d["E_total"])
        rows.append({
            "structure":            name,
            "parent_phase":         d["parent"],
            "n_Mg":                 d["n_Mg"],
            "n_Zn":                 d["n_Zn"],
            "atoms_per_cell":       d["n_Mg"] + d["n_Zn"],
            "E_total_eV":           round(d["E_total"], 4),
            "deltaH_f_eV_per_atom": round(dH, 4),
        })
 
    write_csv("results_antisite.csv",
              ["structure", "parent_phase", "n_Mg", "n_Zn",
               "atoms_per_cell", "E_total_eV", "deltaH_f_eV_per_atom"],
              rows)
 
    print("\nAnti-site defects (eV/atom):")
    for r in rows:
        print(f"  {r['structure']:10s}  parent={r['parent_phase']:8s}  "
              f"DeltaH_f = {r['deltaH_f_eV_per_atom']:+.4f}")
 
 
if __name__ == "__main__":
    main()

Wrote results_antisite.csv

Anti-site defects (eV/atom):
  ZnMg2       parent=MgZn2     DeltaH_f = +0.4040
  Zn2Mg11     parent=Mg2Zn11   DeltaH_f = +0.6212
  Zn4Mg7      parent=Mg4Zn7    DeltaH_f = +0.0104


In [3]:
import csv
 
 
E_MG_BULK = -1.54928
E_ZN_BULK = -1.09020
 
 
HYPOTHETICAL = {
    "MgZn_B2":    {"n_Mg": 1, "n_Zn": 1, "E_total": -2.603480,
                   "prototype": "CsCl-type, Pm-3m"},
    "MgZn2_C11b": {"n_Mg": 2, "n_Zn": 4, "E_total": -5.633427,
                   "prototype": "MoSi2-type, I4/mmm"},
    "Mg2Zn_C11b": {"n_Mg": 4, "n_Zn": 2, "E_total": -4.247121,
                   "prototype": "MoSi2-type, I4/mmm"},
}
 
 
def formation_enthalpy(n_Mg, n_Zn, E_total):
    n = n_Mg + n_Zn
    return (E_total - n_Mg * E_MG_BULK - n_Zn * E_ZN_BULK) / n
 
 
def write_csv(filename, fieldnames, rows):
    with open(filename, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)
    print(f"Wrote {filename}")
 
 
def main():
    rows = []
    for name, d in HYPOTHETICAL.items():
        dH = formation_enthalpy(d["n_Mg"], d["n_Zn"], d["E_total"])
        rows.append({
            "structure":            name,
            "prototype":            d["prototype"],
            "n_Mg":                 d["n_Mg"],
            "n_Zn":                 d["n_Zn"],
            "atoms_per_cell":       d["n_Mg"] + d["n_Zn"],
            "E_total_eV":           round(d["E_total"], 4),
            "deltaH_f_eV_per_atom": round(dH, 4),
        })
 
    write_csv("results_hypothetical.csv",
              ["structure", "prototype", "n_Mg", "n_Zn",
               "atoms_per_cell", "E_total_eV", "deltaH_f_eV_per_atom"],
              rows)
 
    print("\nHypothetical structures (eV/atom):")
    for r in rows:
        print(f"  {r['structure']:12s}  {r['prototype']:25s}  "
              f"DeltaH_f = {r['deltaH_f_eV_per_atom']:+.4f}")
 
 
if __name__ == "__main__":
    main()

Wrote results_hypothetical.csv

Hypothetical structures (eV/atom):
  MgZn_B2       CsCl-type, Pm-3m           DeltaH_f = +0.0180
  MgZn2_C11b    MoSi2-type, I4/mmm         DeltaH_f = +0.3043
  Mg2Zn_C11b    MoSi2-type, I4/mmm         DeltaH_f = +0.6884
